# Storage Write API : upsert 와 `_CHANGE_SEQUENCE_NUMBER` 실습

Python `append_rows` 로 같은 키에 대해 upsert / DELETE 를 보내고, **순서가 뒤바뀌어도 최신 상태가 남는지** 확인한다.

| 시나리오 | 보내는 순서 | 예상 최종 상태 (문서 기준) |
|---|---|---|
| A. 정상 순서 | version 1 → 2 → 3 | `COMPLETED` (version 3) |
| B. 역순 도착 | version 3 → 1 | `COMPLETED` (version 3 유지) |
| C. 같은 version 재전송 | version 3 (status 만 다름) | `CANCELED` (나중에 적재된 행) |
| D. 삭제 후 오래된 UPSERT | version 1 UPSERT → 5 DELETE → 4 UPSERT | 행 없음 (삭제 유지) |

**규칙 메모 (문서 기준, 리허설에서 확인 필요)**
- `_CHANGE_SEQUENCE_NUMBER` 는 16진수 문자열이며 `/` 로 구분해 최대 4개 섹션까지 쓸 수 있다. 여기서는 `version` 을 16자리 16진수로 만든다.
- 한 번이라도 시퀀스를 쓰기 시작했다면 **모든 행에 값을 넣어야** 한다. 섞이면 순서가 예측 불가다.
- `_CHANGE_TYPE`, `_CHANGE_SEQUENCE_NUMBER` 는 테이블 조회에서 읽을 수 없다. 그래서 확인용으로 `version` 컬럼을 따로 저장한다.

**실행 전 준비**
- 노트북 실행 계정에 `BigQuery Data Editor`, `BigQuery Job User`, `BigQuery Read Session User` 권한
- 리전은 **asia-northeast3** 로 통일
- 아래 셀은 **위에서부터 순서대로** 실행한다. 시나리오 셀은 한 번씩만 실행한다.

## 0. 설정

In [ ]:
PROJECT = "YOUR_PROJECT_ID"
DATASET = "write_demo"
TABLE = "orders_seq"
FQ_TABLE = f"{PROJECT}.{DATASET}.{TABLE}"
WRITE_STREAM = f"projects/{PROJECT}/datasets/{DATASET}/tables/{TABLE}/_default"  # 기본 스트림

from google.cloud import bigquery, bigquery_storage_v1
from google.cloud.bigquery_storage_v1 import types, writer
from google.protobuf import descriptor_pb2, descriptor_pool, message_factory

bq = bigquery.Client(project=PROJECT, location="asia-northeast3")
bq_storage = bigquery_storage_v1.BigQueryWriteClient()
print("설정 완료 :", FQ_TABLE)

## 1. 테이블 준비
실습 전용 테이블 `orders_seq` 만 DROP 후 다시 만든다. 다른 테이블은 건드리지 않는다. **한 번만 실행**한다.

In [ ]:
bq.query(f"DROP TABLE IF EXISTS `{FQ_TABLE}`").result()
bq.query(f"""
    CREATE TABLE `{FQ_TABLE}` (
        order_id INT64 PRIMARY KEY NOT ENFORCED,
        status   STRING,
        amount   INT64,
        version  INT64
    )
""").result()
print("테이블 준비 완료 :", FQ_TABLE)

## 2. 행 구조와 쓰기 스트림
`_CHANGE_TYPE`(UPSERT / DELETE)과 `_CHANGE_SEQUENCE_NUMBER` 필드가 upsert 동작을 결정한다. **한 번만 실행**한다.

In [ ]:
F_ = descriptor_pb2.FieldDescriptorProto
row_proto = descriptor_pb2.DescriptorProto(name="OrderSeqRow")
for name, num, typ in [
    ("order_id", 1, F_.TYPE_INT64),
    ("status", 2, F_.TYPE_STRING),
    ("amount", 3, F_.TYPE_INT64),
    ("version", 4, F_.TYPE_INT64),
    ("_CHANGE_TYPE", 5, F_.TYPE_STRING),
    ("_CHANGE_SEQUENCE_NUMBER", 6, F_.TYPE_STRING),
]:
    row_proto.field.add(name=name, number=num, type=typ, label=F_.LABEL_OPTIONAL)

file_proto = descriptor_pb2.FileDescriptorProto(
    name="order_seq.proto", package="write_demo", message_type=[row_proto], syntax="proto2"
)
pool = descriptor_pool.DescriptorPool()
pool.Add(file_proto)
ROW_CLS = message_factory.GetMessageClass(pool.FindMessageTypeByName("write_demo.OrderSeqRow"))

# 첫 요청에 스트림 이름과 행 구조를 담는다
template = types.AppendRowsRequest(
    write_stream=WRITE_STREAM,
    proto_rows=types.AppendRowsRequest.ProtoData(
        writer_schema=types.ProtoSchema(proto_descriptor=row_proto)
    ),
)
STREAM = writer.AppendRowsStream(bq_storage, template)
print("스트림 준비 완료")

## 3. 전송 함수
`send()` 는 행 목록을 한 번의 `append_rows` 로 보내고, 적재가 끝날 때까지 기다린다. 시나리오 셀에서 사용한다.

In [ ]:
def send(changes):
    # changes : [(order_id, change_type, status, amount, version), ...]
    rows = []
    for order_id, change_type, status, amount, version in changes:
        rows.append(
            ROW_CLS(
                order_id=order_id,
                status=status,
                amount=amount,
                version=version,
                _CHANGE_TYPE=change_type,                     # UPSERT 또는 DELETE
                _CHANGE_SEQUENCE_NUMBER=f"{version:016X}",   # 모든 행에 항상 넣는다
            ).SerializeToString()
        )
    request = types.AppendRowsRequest(
        proto_rows=types.AppendRowsRequest.ProtoData(rows=types.ProtoRows(serialized_rows=rows))
    )
    STREAM.send(request).result()  # 적재가 끝날 때까지 기다린다
    print(f"{len(rows)}건 전송 완료")

## 4. 시나리오 A : 정상 순서
version 1 → 2 → 3 을 순서대로 보낸다. **한 번만 실행**한다.

예상 결과 : `order_id = 1` 의 최종 status 는 `COMPLETED` (version 3)

In [ ]:
send([(1, "UPSERT", "PAID", 15000, 1)])
send([(1, "UPSERT", "SHIPPED", 15000, 2)])
send([(1, "UPSERT", "COMPLETED", 15000, 3)])

## 5. 시나리오 B : 역순 도착
version 3 을 먼저, version 1 을 나중에 보낸다. **한 번만 실행**한다.

예상 결과 : `order_id = 2` 의 최종 status 는 `COMPLETED` (version 3 유지)

In [ ]:
send([(2, "UPSERT", "COMPLETED", 20000, 3)])
send([(2, "UPSERT", "PAID", 20000, 1)])

## 6. 시나리오 C : 같은 version 재전송
version 3 을 status 만 바꿔서 다시 보낸다. **한 번만 실행**한다.

예상 결과 : `order_id = 2` 의 최종 status 는 `CANCELED` (나중에 적재된 행, 문서 기준). 실제로 다르면 리허설에서 확인 필요.

In [ ]:
send([(2, "UPSERT", "CANCELED", 20000, 3)])

## 7. 시나리오 D : 삭제 후 오래된 UPSERT
version 1 UPSERT → version 5 DELETE → version 4 UPSERT 순서로 보낸다. **한 번만 실행**한다.

예상 결과 : `order_id = 3` 은 **행이 없다** (삭제가 유지됨). DELETE 반영에 지연이 있을 수 있으니 바로 안 보이면 잠시 뒤 다시 조회한다. (리허설에서 확인 필요)

In [ ]:
send([(3, "UPSERT", "PAID", 9000, 1)])
send([(3, "DELETE", "", 0, 5)])
send([(3, "UPSERT", "SHIPPED", 9000, 4)])

## 8. 결과 비교 (SQL)
시나리오 실행이 모두 끝난 뒤 콘솔 쿼리 편집기에서 실행한다.

```sql
SELECT order_id, status, amount, version
FROM `YOUR_PROJECT_ID.write_demo.orders_seq`
ORDER BY order_id;
```

예상 결과

| order_id | 시나리오 | 예상 status | 예상 version |
|---|---|---|---|
| 1 | A 정상 순서 | COMPLETED | 3 |
| 2 | B·C 역순 도착 / 재전송 | CANCELED | 3 |
| 3 | D 삭제 후 오래된 UPSERT | (행 없음) | - |

> 2번 주문의 `CANCELED` 는 문서 기준 예상이다. 리허설에서 `COMPLETED` 로 남는다면 문서 설명과 다르니 결과를 그대로 기록한다.